# 06 - Operating-threshold optimization

## Why this notebook exists

A churn model produces probabilities, but the business must choose a decision threshold. A customer is flagged only when `predicted_probability >= threshold`.

Do not choose a threshold from the final test set. This notebook selects it from out-of-fold predictions on the training data, then evaluates the result once on the untouched test set.

The referenced paper reports 92.3% accuracy on a different expanded dataset that includes technical-ticket information. The public IBM Telco dataset in this project does not contain that feature, so we report honest results for the available data rather than claiming the paper's number.

## Load the selected model

Notebook 02 saves all calibrated models, the independent test set, and the validation ranking. The champion is selected from validation PR-AUC, not from test accuracy.

In [ ]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS = ROOT / 'artifacts'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)

with open(ARTIFACTS / 'benchmark_models.pkl', 'rb') as handle:
    bundle = pickle.load(handle)

champion_name = bundle['results'].sort_values('validation_pr_auc_mean', ascending=False).iloc[0]['model']
champion_model = bundle['models'][champion_name]
X_test = bundle['X_test']
y_test = bundle['y_test']

# Reconstruct the same training rows from notebook 02 for honest OOF threshold tuning.
data = pd.read_csv(ROOT / 'data' / 'telco_clean_32col.csv')
X = data.drop(columns='Churn')
y = data['Churn'].astype(int)
training_index = X.index.difference(X_test.index)
X_train = X.loc[training_index]
y_train = y.loc[training_index]
validation_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print('Champion:', champion_name)
print('Training rows:', len(X_train), 'Test rows:', len(X_test))

## Select a threshold from out-of-fold predictions

F1 is used as the default selection criterion because it balances precision and recall. Accuracy is displayed, but it is not the selection target because churn is imbalanced.

In [ ]:
oof_probability = cross_val_predict(
    champion_model,
    X_train,
    y_train,
    cv=validation_cv,
    method='predict_proba',
    n_jobs=-1,
)[:, 1]

rows = []
for threshold in np.arange(0.05, 0.96, 0.01):
    prediction = (oof_probability >= threshold).astype(int)
    rows.append({
        'threshold': threshold,
        'accuracy': accuracy_score(y_train, prediction),
        'balanced_accuracy': balanced_accuracy_score(y_train, prediction),
        'precision': precision_score(y_train, prediction, zero_division=0),
        'recall': recall_score(y_train, prediction, zero_division=0),
        'f1': f1_score(y_train, prediction, zero_division=0),
    })

threshold_table = pd.DataFrame(rows)
decision_threshold = float(threshold_table.loc[threshold_table['f1'].idxmax(), 'threshold'])
display(threshold_table.sort_values('f1', ascending=False).head(10))
print('Selected threshold:', decision_threshold)

## Visualize the accuracy-recall trade-off

A threshold that raises plain accuracy can reduce recall by labeling fewer people as churners. The selected vertical line shows the F1-balanced operating point.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for metric in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1']:
    ax.plot(threshold_table['threshold'], threshold_table[metric], label=metric)

ax.axvline(decision_threshold, color='black', linestyle='--', label='selected threshold')
ax.set_title('Out-of-fold threshold trade-off')
ax.set_xlabel('Churn probability threshold')
ax.set_ylabel('Score')
ax.legend(ncol=2)
ax.grid(alpha=.25)
plt.tight_layout()
plt.savefig(FIGURES / 'threshold_tradeoff.png', dpi=160)
plt.show()

## One-time test-set evaluation

The test set is used only here, after the threshold was selected. This is the number to put in the report.

In [ ]:
test_probability = champion_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= decision_threshold).astype(int)

test_metrics = pd.Series({
    'threshold': decision_threshold,
    'accuracy': accuracy_score(y_test, test_prediction),
    'balanced_accuracy': balanced_accuracy_score(y_test, test_prediction),
    'precision': precision_score(y_test, test_prediction, zero_division=0),
    'recall': recall_score(y_test, test_prediction, zero_division=0),
    'f1': f1_score(y_test, test_prediction, zero_division=0),
})
display(test_metrics.to_frame('test_value'))

operating_policy = {
    'model_name': champion_name,
    'decision_threshold': decision_threshold,
    'selection_metric': 'out_of_fold_f1',
    'seed': SEED,
}
(ARTIFACTS / 'operating_threshold.json').write_text(
    json.dumps(operating_policy, indent=2),
    encoding='utf-8',
)
print('Saved:', ARTIFACTS / 'operating_threshold.json')

## Report wording

Report the selected threshold together with accuracy, balanced accuracy, precision, recall, and F1. Do not claim the 92.3% accuracy reported in Zerine et al. (2026), because that study used additional technical-ticket fields unavailable in the public IBM Telco release.